# Advanced Python for Battery Engineers with PyBaMM — Solutions

> **Environment note:** these notebooks need a Python environment with a working
> PyBaMM install (`pip install pybamm numpy pandas scipy matplotlib ipykernel`).
> All simulations below use `pybamm.lithium_ion` models with the built-in
> `"Chen2020"` parameter set and run in well under a second each, so nothing
> here should take more than a few seconds to execute.

Every problem below is fully solved and has been executed top-to-bottom.

In [1]:
%matplotlib inline
import time
import itertools
from functools import partial, lru_cache
from contextlib import contextmanager
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

import pybamm

pybamm.set_logging_level("WARNING")

## 1. Object-Oriented Battery Models

Three problems, increasing in scope: a validated property, a small numerical method on a class, and a dataclass wired into a real PyBaMM simulation.

#### Problem 1: A validated `Cell` class

Complete the `Cell` class below:
- the `capacity_ah` setter must raise `ValueError` if `value <= 0`;
- `energy_wh` must return `capacity_ah * nominal_voltage`.

In [2]:
class Cell:
    def __init__(self, capacity_ah, nominal_voltage):
        self.capacity_ah = capacity_ah
        self.nominal_voltage = nominal_voltage

    @property
    def capacity_ah(self):
        return self._capacity_ah

    @capacity_ah.setter
    def capacity_ah(self, value):
        if value <= 0:
            raise ValueError("capacity_ah must be positive")
        self._capacity_ah = value

    @property
    def energy_wh(self):
        return self.capacity_ah * self.nominal_voltage

In [3]:
cell = Cell(5.0, 3.7)
print(f"Energy: {cell.energy_wh:.2f} Wh")

try:
    Cell(-2.0, 3.7)
    print("No error raised (unexpected)")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

Energy: 18.50 Wh
Correctly raised ValueError: capacity_ah must be positive


#### Problem 2: RC-branch impedance

Implement `RCPair.impedance(omega)`, returning the complex impedance of a resistor
`r1` in parallel with a capacitor `c1`, in series with `r0`:

$$Z(\omega) = R_0 + \dfrac{R_1}{1 + j \omega R_1 C_1}$$

In [4]:
class RCPair:
    def __init__(self, r0, r1, c1):
        self.r0 = r0
        self.r1 = r1
        self.c1 = c1

    def impedance(self, omega):
        return self.r0 + self.r1 / (1 + 1j * omega * self.r1 * self.c1)

In [5]:
branch = RCPair(r0=0.01, r1=0.02, c1=2000)
tau = branch.r1 * branch.c1
z = branch.impedance(omega=1 / tau)
print(f"Impedance at omega=1/tau: {z:.5f} ohm")
print(f"|Z| = {abs(z):.5f} ohm, phase = {np.angle(z, deg=True):.2f} deg")

Impedance at omega=1/tau: 0.02000-0.01000j ohm
|Z| = 0.02236 ohm, phase = -26.57 deg


#### Problem 3: `CellDesign` dataclass driving a real simulation

Complete `CellDesign.to_parameter_updates()` so it returns a dict mapping
`"Negative electrode thickness [m]"` and `"Positive electrode thickness [m]"` to
this design's values. `final_voltage` (given) uses it to run a 1-hour SPM discharge.

In [6]:
@dataclass
class CellDesign:
    name: str
    negative_electrode_thickness_m: float
    positive_electrode_thickness_m: float

    def to_parameter_updates(self):
        return {
            "Negative electrode thickness [m]": self.negative_electrode_thickness_m,
            "Positive electrode thickness [m]": self.positive_electrode_thickness_m,
        }


def final_voltage(design):
    params = pybamm.ParameterValues("Chen2020")
    params.update(design.to_parameter_updates())
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 3600])
    return sol["Terminal voltage [V]"].data[-1]

In [7]:
baseline = CellDesign("baseline", 8.52e-05, 7.56e-05)
thicker_positive = CellDesign("thicker_positive", 8.52e-05, 7.56e-05 * 1.3)

for design in (baseline, thicker_positive):
    v_end = final_voltage(design)
    print(f"{design.name}: V_end after 1 h = {v_end:.4f} V")

baseline: V_end after 1 h = 2.5000 V


thicker_positive: V_end after 1 h = 2.5000 V


## 2. Functional Python for Parameter Functions

Three problems building families of PyBaMM parameter functions: a plain closure, an Arrhenius closure, and `functools.partial`.

#### Problem 1: A constant-current closure factory

Complete `make_constant_current(magnitude_a)` so it returns a function of `t` that
always evaluates to `magnitude_a` (keep the `+ 0.0 * t` so the result has the right
symbolic shape).

In [8]:
def make_constant_current(magnitude_a):
    def current_fn(t):
        return magnitude_a + 0.0 * t
    return current_fn


def run_at_current(magnitude_a):
    params = pybamm.ParameterValues("Chen2020")
    params["Current function [A]"] = make_constant_current(magnitude_a)
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    return sim.solve([0, 1800])

In [9]:
for magnitude in (2.5, 5.0):
    sol = run_at_current(magnitude)
    print(
        f"I={magnitude} A: V_end = {sol['Terminal voltage [V]'].data[-1]:.4f} V, "
        f"capacity = {sol['Discharge capacity [A.h]'].data[-1]:.4f} Ah"
    )

I=2.5 A: V_end = 3.8833 V, capacity = 1.2500 Ah


I=5.0 A: V_end = 3.5684 V, capacity = 2.5000 Ah


#### Problem 2: Arrhenius diffusivity closure

Complete `make_arrhenius_diffusivity` so the returned `D(sto, T)` implements
$D(T) = D_{ref} \exp\!\left(-\dfrac{E_a}{R}\left(\dfrac{1}{T}-\dfrac{1}{T_{ref}}\right)\right)$
using `pybamm.exp` (not `np.exp` — `T` is a symbolic PyBaMM expression here).

In [10]:
def make_arrhenius_diffusivity(D_ref, activation_energy_j_per_mol, T_ref=298.15):
    gas_constant = 8.314

    def D(sto, T):
        return D_ref * pybamm.exp(
            -activation_energy_j_per_mol / gas_constant * (1 / T - 1 / T_ref)
        )

    return D

In [11]:
D_low = make_arrhenius_diffusivity(D_ref=3.3e-14, activation_energy_j_per_mol=20000)
D_high = make_arrhenius_diffusivity(D_ref=3.3e-14, activation_energy_j_per_mol=50000)

for label, D_fn in [("low Ea", D_low), ("high Ea", D_high)]:
    params = pybamm.ParameterValues("Chen2020")
    params["Negative electrode diffusivity [m2.s-1]"] = D_fn
    params["Ambient temperature [K]"] = 278.15
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 1800])
    print(f"{label}: V_end at cold start = {sol['Terminal voltage [V]'].data[-1]:.4f} V")

low Ea: V_end at cold start = 3.5200 V


high Ea: V_end at cold start = 3.5200 V


#### Problem 3: `functools.partial` profile variants

Complete `sinusoidal_current(t, magnitude, period_s)` to implement
`magnitude * (1 + sin(2*pi*t/period_s))` using `pybamm.sin`. The test cell uses
`functools.partial` to build two named variants from it.

In [12]:
def sinusoidal_current(t, magnitude, period_s):
    return magnitude * (1 + pybamm.sin(2 * np.pi * t / period_s))


def run_profile(current_fn):
    params = pybamm.ParameterValues("Chen2020")
    params["Current function [A]"] = current_fn
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    return sim.solve([0, 1800])

In [13]:
aggressive = partial(sinusoidal_current, magnitude=6.0, period_s=200)
gentle = partial(sinusoidal_current, magnitude=3.0, period_s=900)

for label, profile in [("aggressive", aggressive), ("gentle", gentle)]:
    sol = run_profile(profile)
    print(f"{label}: capacity discharged = {sol['Discharge capacity [A.h]'].data[-1]:.4f} Ah")

aggressive: capacity discharged = 3.0001 Ah


gentle: capacity discharged = 1.4999 Ah


## 3. Iterators, Generators, Decorators and Context Managers

Three problems: a step-generator for a cycling protocol, a grid sweep built with `itertools.product`, and a timing context manager.

#### Problem 1: A CCCV cycling-protocol generator

Complete `cccv_cycle_steps` so it `yield`s **two** strings per cycle:
`f"Charge at {charge_c_rate}C until 4.2 V"` and
`f"Discharge at {discharge_c_rate}C until 3.0 V"`.

In [14]:
def cccv_cycle_steps(n_cycles, charge_c_rate=0.5, discharge_c_rate=1.0):
    for _ in range(n_cycles):
        yield f"Charge at {charge_c_rate}C until 4.2 V"
        yield f"Discharge at {discharge_c_rate}C until 3.0 V"


In [15]:
steps = list(cccv_cycle_steps(3))
experiment = pybamm.Experiment(steps)
print(f"Number of steps generated: {len(steps)}")
print(f"First step: {steps[0]!r}")

Number of steps generated: 6
First step: 'Charge at 0.5C until 4.2 V'


#### Problem 2: A sweep grid with `itertools.product`

Complete `run_grid` to loop over `itertools.product(c_rates, temperatures)` instead of
the empty placeholder loop, running one short SPM simulation per combination.

In [16]:
def run_grid(c_rates, temperatures):
    results = []
    for c_rate, T in itertools.product(c_rates, temperatures):
        params = pybamm.ParameterValues("Chen2020")
        nominal_capacity = params["Nominal cell capacity [A.h]"]
        params["Current function [A]"] = nominal_capacity * c_rate
        params["Ambient temperature [K]"] = T
        sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
        sol = sim.solve([0, 600])
        results.append({"c_rate": c_rate, "temperature_K": T, "v_end": sol["Terminal voltage [V]"].data[-1]})
    return results

In [17]:
results = run_grid([0.5, 1.0, 2.0], [278.15, 298.15])
print(f"Number of combinations: {len(results)}")
for r in results:
    print(f"C-rate={r['c_rate']}, T={r['temperature_K']}K -> V_end={r['v_end']:.4f} V")

Number of combinations: 6
C-rate=0.5, T=278.15K -> V_end=3.9723 V
C-rate=0.5, T=298.15K -> V_end=4.0164 V
C-rate=1.0, T=278.15K -> V_end=3.8197 V
C-rate=1.0, T=298.15K -> V_end=3.8679 V
C-rate=2.0, T=278.15K -> V_end=3.5172 V
C-rate=2.0, T=298.15K -> V_end=3.5696 V


#### Problem 3: A `timer` context manager

Complete `timer` so entering the `with` block records the start time, and leaving it
(via `finally`, so it always runs) prints `f"{label}: {elapsed:.4f} s"`.

In [18]:
@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label}: {elapsed:.4f} s")

In [19]:
for label, Model in [("SPM", pybamm.lithium_ion.SPM), ("DFN", pybamm.lithium_ion.DFN)]:
    with timer(label):
        pybamm.Simulation(Model()).solve([0, 3600])

SPM: 0.2585 s


DFN: 0.5937 s


## 4. NumPy Vectorization for Post-Processing

Three problems on vectorized post-processing of a `pybamm.Solution`: capacity integration, dQ/dV, and RMSE between two curves.

#### Problem 1: Discharge capacity via `np.trapezoid`

Complete `discharge_capacity_ah` to integrate current over time (converted to hours)
with `np.trapezoid`.

In [20]:
def discharge_capacity_ah(solution):
    t = solution["Time [s]"].data
    current = solution["Current [A]"].data
    return np.trapezoid(current, t) / 3600

In [21]:
sol = pybamm.Simulation(pybamm.lithium_ion.SPM()).solve([0, 3600])
cap = discharge_capacity_ah(sol)
print(f"Integrated capacity: {cap:.4f} Ah")
print(f"PyBaMM's own tracked capacity: {sol['Discharge capacity [A.h]'].data[-1]:.4f} Ah")

Integrated capacity: 0.6806 Ah
PyBaMM's own tracked capacity: 0.6806 Ah


#### Problem 2: Incremental capacity analysis

Complete `dqdv` to return `np.gradient(capacity, voltage)`, suppressing divide-by-zero
warnings with `np.errstate`.

In [22]:
def dqdv(voltage, capacity):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.gradient(capacity, voltage)

In [23]:
sol = pybamm.Simulation(pybamm.lithium_ion.SPM()).solve([0, 3600])
v = sol["Terminal voltage [V]"].data
q = sol["Discharge capacity [A.h]"].data
dqdv_curve = dqdv(v, q)
print(f"dQ/dV shape: {dqdv_curve.shape}")
print(f"Max |dQ/dV|: {np.nanmax(np.abs(dqdv_curve)):.4f} Ah/V")

dQ/dV shape: (51,)
Max |dQ/dV|: 3.8160 Ah/V


#### Problem 3: RMSE between two voltage curves

Complete `rmse_voltage`: interpolate both solutions' voltage onto `t_common` with
`np.interp`, then return the RMSE between them.

In [24]:
def rmse_voltage(sol_a, sol_b, n_points=100):
    t_a, v_a = sol_a["Time [s]"].data, sol_a["Terminal voltage [V]"].data
    t_b, v_b = sol_b["Time [s]"].data, sol_b["Terminal voltage [V]"].data
    t_common = np.linspace(0, min(t_a[-1], t_b[-1]), n_points)
    v_a_common = np.interp(t_common, t_a, v_a)
    v_b_common = np.interp(t_common, t_b, v_b)
    return np.sqrt(np.mean((v_a_common - v_b_common) ** 2))

In [25]:
sol_spm = pybamm.Simulation(pybamm.lithium_ion.SPM()).solve([0, 3600])
sol_dfn = pybamm.Simulation(pybamm.lithium_ion.DFN()).solve([0, 3600])
error = rmse_voltage(sol_spm, sol_dfn)
print(f"RMSE between SPM and DFN voltage: {error * 1000:.2f} mV")

RMSE between SPM and DFN voltage: 19.41 mV


## 5. Pandas for Parameter Sweeps and Tidy Data

Three problems: building a sweep DataFrame, pivoting a 2D sweep, and merging against reference data.

#### Problem 1: Sweep results into a DataFrame

Complete `sweep_c_rates` so each loop iteration appends a row dict with keys
`c_rate`, `v_end`, `capacity_ah` from a short SPM simulation at that C-rate.

In [26]:
def sweep_c_rates(c_rates):
    rows = []
    for c_rate in c_rates:
        params = pybamm.ParameterValues("Chen2020")
        nominal_capacity = params["Nominal cell capacity [A.h]"]
        params["Current function [A]"] = nominal_capacity * c_rate
        sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
        sol = sim.solve([0, 3600 / c_rate])
        rows.append(
            {
                "c_rate": c_rate,
                "v_end": sol["Terminal voltage [V]"].data[-1],
                "capacity_ah": sol["Discharge capacity [A.h]"].data[-1],
            }
        )
    return pd.DataFrame(rows)

In [27]:
df = sweep_c_rates([0.5, 1.0, 2.0])
print(df)

   c_rate     v_end  capacity_ah
0     0.5  2.583528     5.000000
1     1.0  2.500000     4.955361
2     2.0  2.500000     4.822208


#### Problem 2: Pivoting a 2D sweep

The data-collection loop is given. Complete the final line to pivot `df` into a table
with `c_rate` as the index, `temperature_K` as columns, and `capacity_ah` as values.

In [28]:
def capacity_table(c_rates, temperatures):
    rows = []
    for c_rate, T in itertools.product(c_rates, temperatures):
        params = pybamm.ParameterValues("Chen2020")
        nominal_capacity = params["Nominal cell capacity [A.h]"]
        params["Current function [A]"] = nominal_capacity * c_rate
        params["Ambient temperature [K]"] = T
        sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
        sol = sim.solve([0, 3600 / c_rate])
        rows.append({"c_rate": c_rate, "temperature_K": T, "capacity_ah": sol["Discharge capacity [A.h]"].data[-1]})
    df = pd.DataFrame(rows)
    return df.pivot(index="c_rate", columns="temperature_K", values="capacity_ah")

In [29]:
table = capacity_table([0.5, 1.0, 2.0], [278.15, 298.15])
print(table)

temperature_K    278.15    298.15
c_rate                           
0.5            5.000000  5.000000
1.0            4.940936  4.955361
2.0            4.802143  4.822208


#### Problem 3: Percent error against reference data

`sim_df` and `reference_df` are already merged. Add a `"pct_error"` column equal to
`100 * (capacity_ah_sim - capacity_ah_ref) / capacity_ah_ref`, then return the result
sorted by `pct_error` descending.

In [30]:
def add_percent_error(sim_df, reference_df):
    merged = sim_df.merge(reference_df, on="c_rate", suffixes=("_sim", "_ref"))
    merged["pct_error"] = (
        100 * (merged["capacity_ah_sim"] - merged["capacity_ah_ref"]) / merged["capacity_ah_ref"]
    )
    return merged.sort_values("pct_error", ascending=False)

In [31]:
sim_df = pd.DataFrame({"c_rate": [0.5, 1.0, 2.0], "capacity_ah": [5.05, 4.93, 4.65]})
reference_df = pd.DataFrame({"c_rate": [0.5, 1.0, 2.0], "capacity_ah": [5.00, 4.97, 4.80]})
result = add_percent_error(sim_df, reference_df)
print(result)

   c_rate  capacity_ah_sim  capacity_ah_ref  pct_error
0     0.5             5.05             5.00   1.000000
1     1.0             4.93             4.97  -0.804829
2     2.0             4.65             4.80  -3.125000


## 6. Optimization and Design-Space Exploration (Capstone)

Three problems building up to the capstone: the objective function, the `scipy.optimize` call, and the full `CellDesignOptimizer` class.

#### Problem 1: An energy-density objective function

Complete `energy_density_wh_per_kg`: integrate `voltage * current` over time with
`np.trapezoid` to get energy in Wh, divide by `thickness_m * mass_scale_kg_per_m`.

In [32]:
def energy_density_wh_per_kg(thickness_m, mass_scale_kg_per_m=5000):
    params = pybamm.ParameterValues("Chen2020")
    params["Positive electrode thickness [m]"] = thickness_m
    sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
    sol = sim.solve([0, 3600])
    t = sol["Time [s]"].data
    v = sol["Terminal voltage [V]"].data
    i = sol["Current [A]"].data
    energy_wh = np.trapezoid(v * i, t) / 3600
    mass_kg = thickness_m * mass_scale_kg_per_m
    return energy_wh / mass_kg

In [33]:
for thickness in (50e-6, 75e-6, 100e-6):
    ed = energy_density_wh_per_kg(thickness)
    print(f"thickness={thickness * 1e6:.0f} um -> energy density = {ed:.3f} Wh/kg (proxy)")

thickness=50 um -> energy density = 50.748 Wh/kg (proxy)


thickness=75 um -> energy density = 46.858 Wh/kg (proxy)


thickness=100 um -> energy density = 36.136 Wh/kg (proxy)


#### Problem 2: Optimizing thickness with `scipy.optimize`

Complete `optimize_thickness` using `minimize_scalar(..., method="bounded")` to
*maximize* `energy_density_wh_per_kg` (i.e. minimize its negative) within `bounds`,
reusing the function from Problem 1.

In [34]:
def optimize_thickness(bounds=(40e-6, 100e-6)):
    result = minimize_scalar(
        lambda x: -energy_density_wh_per_kg(x), bounds=bounds, method="bounded"
    )
    return result.x

In [35]:
best_thickness = optimize_thickness()
best_ed = energy_density_wh_per_kg(best_thickness)
print(f"Optimal thickness: {best_thickness * 1e6:.2f} um")
print(f"Energy density at optimum: {best_ed:.3f} Wh/kg (proxy)")

Optimal thickness: 62.92 um
Energy density at optimum: 52.235 Wh/kg (proxy)


#### Problem 3: The `CellDesignOptimizer` capstone class

Complete `CellDesignOptimizer.objective`: compute the energy density (as in
Problem 1) from `sol`, then subtract `1000 * (self.min_capacity_ah - capacity_ah)`
as a penalty *only if* the discharge capacity is below `min_capacity_ah`.

In [36]:
class CellDesignOptimizer:
    def __init__(self, min_capacity_ah=4.5):
        self.min_capacity_ah = min_capacity_ah
        self._cache = {}

    def _simulate(self, thickness_m):
        if thickness_m not in self._cache:
            params = pybamm.ParameterValues("Chen2020")
            params["Positive electrode thickness [m]"] = thickness_m
            sim = pybamm.Simulation(pybamm.lithium_ion.SPM(), parameter_values=params)
            self._cache[thickness_m] = sim.solve([0, 3600])
        return self._cache[thickness_m]

    def objective(self, thickness_m):
        sol = self._simulate(thickness_m)
        t = sol["Time [s]"].data
        power_w = sol["Terminal voltage [V]"].data * sol["Current [A]"].data
        energy_wh = np.trapezoid(power_w, t) / 3600
        mass_kg = thickness_m * 5000
        energy_density = energy_wh / mass_kg

        capacity_ah = sol["Discharge capacity [A.h]"].data[-1]
        if capacity_ah < self.min_capacity_ah:
            energy_density -= 1000 * (self.min_capacity_ah - capacity_ah)
        return energy_density

    def optimize(self, bounds=(40e-6, 100e-6)):
        result = minimize_scalar(lambda x: -self.objective(x), bounds=bounds, method="bounded")
        return result.x

In [37]:
optimizer = CellDesignOptimizer(min_capacity_ah=4.0)
best = optimizer.optimize()
print(f"Optimizer chose thickness: {best * 1e6:.2f} um")
print(f"Cache size after optimization: {len(optimizer._cache)} unique evaluations")

Optimizer chose thickness: 62.92 um
Cache size after optimization: 5 unique evaluations
